🧠 AutoGen Assessment Scenario
🎯 Title: Multi-Agent AI System for Automated Code Review & Execution
🔹 Scenario Overview

You are part of an AI engineering team building an AutoGen-based developer assistant.

The system should:

Accept a user problem statement
Generate code
Review the code
Execute it
Provide feedback and corrections

This must be implemented using multiple collaborating agents.

🔹 Problem Statement

A user gives the following input:

“Write a Python function to find the second largest number in a list and handle edge cases.”

Your AutoGen system should:

Generate correct code
Validate logic
Execute the code
Handle errors
Provide final response
🔹 Requirements
✅ 1. Agent Design (MANDATORY)

Create at least 3 agents:

🔸 Coder Agent
Generates Python code
Should follow best practices
Must handle edge cases
🔸 Reviewer Agent
Reviews code for:
Bugs
Logic errors
Edge cases
Suggests improvements
🔸 Executor Agent (UserProxy)
Runs the code
Returns output/errors
🔹 2. Agent Collaboration Flow

Design the flow such that:

User → Coder → Reviewer → Executor → Reviewer (if error) → Final Output

👉 The system should not stop after one step
👉 It must iterate until correct output is achieved

🔹 3. Functional Expectations

Your system must:

✔ Handle incorrect code generated initially
✔ Ensure reviewer gives structured feedback
✔ Prevent infinite loops
✔ Show final clean output

🔹 4. Technical Constraints
Use AutoGen framework
Use LLM-based agents
Enable code execution
Add termination condition
🔹 5. Edge Cases to Handle

Test your system with:

Empty list
List with one element
Duplicate values
Negative numbers

In [1]:
%pip install pyautogen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 5.7 MB/s eta 0:00:00


In [9]:
# install (run once)
import sys, subprocess
def install(p): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])
install("pyautogen==0.2.25"); install("openai==1.3.8"); install("httpx==0.27.0")

In [3]:
# install once
import sys, subprocess
def install(p): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])
install("pyautogen==0.2.25"); install("openai==1.3.8"); install("httpx==0.27.0")

import os, autogen
from google.colab import userdata

# key
key = userdata.get("groq_api_key")
if not key: raise ValueError("Missing key")
os.environ["GROQ_API_KEY"] = key

# config
config_list = [{
    "model": "openai/gpt-oss-20b",
    "api_key": os.environ["GROQ_API_KEY"],
    "base_url": "https://api.groq.com/openai/v1",
}]

llm_config = {
    "config_list": config_list,
    "temperature": 0.2,   #  reduce randomness
    "max_tokens": 600,    #  prevent overflow
    "top_p": 1,
}

# coder
coder = autogen.AssistantAgent(
    name="Coder",
    llm_config=llm_config,
    system_message="Write correct Python code only. Handle all edge cases."
)

# reviewer
reviewer = autogen.AssistantAgent(
    name="Reviewer",
    llm_config=llm_config,
    system_message=(
        "Review code.\n"
        "If correct say: FINAL APPROVED\n"
        "Else give fixes only."
    )
)

# executor
executor = autogen.UserProxyAgent(
    name="Executor",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=4,  #stop infinite loop
    code_execution_config={"work_dir": "coding", "use_docker": False},
    is_termination_msg=lambda m: "FINAL APPROVED" in (m.get("content") or "")
)

# correct flow: coder → reviewer → executor
groupchat = autogen.GroupChat(
    agents=[coder, reviewer, executor],
    messages=[],
    max_round=5,
    speaker_selection_method="round_robin"
)

manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=llm_config)

# run
executor.initiate_chat(
    manager,
    message="Write Python function for second largest number with edge cases."
)

Executor (to chat_manager):

Write Python function for second largest number with edge cases.

--------------------------------------------------------------------------------
Coder (to chat_manager):

```python
def second_largest(nums):
    """
    Return the second largest *distinct* number from an iterable.

    Parameters
    ----------
    nums : iterable
        An iterable of numeric values (int or float).  The iterable may be a list,
        tuple, generator

--------------------------------------------------------------------------------
Reviewer (to chat_manager):

**Fixes**

```python
def second_largest(nums):
    """
    Return the second largest *distinct* number from an iterable.

    Parameters
    ----------
    nums : iterable
        An iterable of numeric values (int or float).  The iterable may be a
        list, tuple, generator, or any other iterable.

    Returns
    -------
    float or int
        The second largest distinct number in the iterable.

    Raises


ChatResult(chat_id=None, chat_history=[{'content': 'Write Python function for second largest number with edge cases.', 'role': 'assistant'}, {'content': '```python\ndef second_largest(nums):\n    """\n    Return the second largest *distinct* number from an iterable.\n\n    Parameters\n    ----------\n    nums : iterable\n        An iterable of numeric values (int or float).  The iterable may be a list,\n        tuple, generator', 'name': 'Coder', 'role': 'user'}, {'content': '**Fixes**\n\n```python\ndef second_largest(nums):\n    """\n    Return the second largest *distinct* number from an iterable.\n\n    Parameters\n    ----------\n    nums : iterable\n        An iterable of numeric values (int or float).  The iterable may be a\n        list, tuple, generator, or any other iterable.\n\n    Returns\n    -------\n    float or int\n        The second largest distinct number in the iterable.\n\n    Raises\n    ------\n    ValueError\n        If the iterable contains fewer than two distin

In [ ]:
import nbformat
import os

# Get the current notebook's filename
notebook_name = 'day23Part2.ipynb'
fixed_notebook_name = notebook_name.replace('.ipynb', '_cleaned.ipynb')

if not os.path.exists(notebook_name):
    print(f'Error: The notebook file {notebook_name} was not found.')
    print('Please upload the notebook file you wish to fix to your environment or provide the correct path.')
else:
    with open(notebook_name, 'r', encoding='utf-8') as f:
        nb = nbformat.read(f, as_version=4)
    # Remove all metadata
    nb['metadata'] = {}
    with open(fixed_notebook_name, 'w', encoding='utf-8') as f:
        nbformat.write(nb, f)
    print(f'Fixed notebook saved to {fixed_notebook_name}!')